# 06 - Build Final PC Members

In this notebook, I create the final visible PC data used by downstream
notebooks.

The source specific scrape is `researchr_pc_members.parquet`. The final file is
`pc_members.parquet`. I keep the raw `researchr_id`, but I use
`canonical_researchr_id` as the stable researcher identifier.


## 1 - Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd


In [2]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from project_setup import setup_project

setup = setup_project()
project_folder = setup.project_folder
PROJECT = project_folder
repo = project_folder
config_path = setup.config_path
project_config = setup.project_config

run_mode = setup.run_mode
inputs_config = setup.inputs
outputs_config = setup.outputs
openalex_config = setup.openalex

allow_network = setup.allow_network
use_existing_data = setup.use_existing_data
overwrite_data = setup.overwrite_data
overwrite_artifacts = setup.overwrite_artifacts
openalex_sample_limit = setup.openalex_sample_limit
openalex_sample_include_work_ids = setup.openalex_sample_include_work_ids

step_1_data_dir = project_folder / "step_1_data"
step_1_artifacts_dir = project_folder / "step_1_artifacts"
prepared_dir = step_1_data_dir / "prepared"
intermediate_dir = step_1_data_dir / "intermediate"
summary_tables_dir = step_1_artifacts_dir / "summary_tables"
dependency_tables_dir = step_1_artifacts_dir / "dependency_tables"
check_tables_dir = step_1_artifacts_dir / "check_tables"

intermediate_dir.mkdir(parents=True, exist_ok=True)
prepared_dir.mkdir(parents=True, exist_ok=True)
summary_tables_dir.mkdir(parents=True, exist_ok=True)
dependency_tables_dir.mkdir(parents=True, exist_ok=True)
check_tables_dir.mkdir(parents=True, exist_ok=True)

print(project_folder)
print(f"Run mode: {run_mode}")


/Users/endersari/2026-02-citations-vs-pc-memberships
Run mode: fast


## 2 - Load Visible PC Source

In [3]:
pc_source = pd.read_parquet(intermediate_dir / "researchr_pc_members.parquet")

print(pc_source.shape)
display(pc_source.head())


(2180, 12)


,conference,year,source,name,role,affiliation,country,person_url,researchr_id,canonical_researchr_id,source_url,final_url
0,ICFP,2017,Researchr PC,Adam Chlipala,PC Member,"Massachusetts Institute of Technology, USA",United States,https://icfp17.sigplan.org/profile/adamchlipala,adamchlipala,adamchlipala,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
1,ICFP,2017,Researchr PC,Alan Jeffrey,PC Member,Mozilla Research,United States,https://icfp17.sigplan.org/profile/alanjeffrey1,alanjeffrey1,alanjeffrey1,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
2,ICFP,2017,Researchr PC,Alexandra Silva,PC Member,University College London,United Kingdom,https://icfp17.sigplan.org/profile/alexandrasilva,alexandrasilva,alexandrasilva,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
3,ICFP,2017,Researchr PC,Ben Lippmeier,PC Member,Digital Asset / UNSW Australia,,https://icfp17.sigplan.org/profile/benlippmeier,benlippmeier,benlippmeier,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
4,ICFP,2017,Researchr PC,Beta Ziliani,PC Member,"FAMAF, UNC and CONICET",Argentina,https://icfp17.sigplan.org/profile/betaziliani,betaziliani,betaziliani,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...


## 3 - Build Final Roster

In [4]:
required_columns = {
    "conference", "year", "source", "name", "role", "affiliation", "country",
    "person_url", "researchr_id", "canonical_researchr_id", "source_url",
    "final_url",
}

missing_columns = sorted(required_columns - set(pc_source.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

pc_members = pc_source.copy()

researcher_identity_merges = pd.DataFrame([
    {
        "source_canonical_researchr_id": "guoqingharryxu",
        "canonical_researchr_id": "harryxu",
        "canonical_name": "Guoqing Harry Xu",
        "reason": "Same researcher listed as Harry Xu on PLDI 2018 and Guoqing Harry Xu on OOPSLA 2019.",
    },
    {
        "source_canonical_researchr_id": "vikashmansinghka1",
        "canonical_researchr_id": "vikashmansinghka",
        "canonical_name": "Vikash K. Mansinghka",
        "reason": "Same researcher listed as Vikash Mansinghka and Vikash K. Mansinghka; both map to the same OpenAlex Author ID.",
    },
    {
        "source_canonical_researchr_id": "corinapasareanu",
        "canonical_researchr_id": "corinaspasareanu",
        "canonical_name": "Corina S. Pasareanu",
        "reason": "Same researcher listed as Corina Pasareanu and Corina S. Pasareanu; both map to the same OpenAlex Author ID.",
    },
    {
        "source_canonical_researchr_id": "corinaspasareanu1",
        "canonical_researchr_id": "corinaspasareanu",
        "canonical_name": "Corina S. Pasareanu",
        "reason": "Same researcher listed with accented and unaccented Corina S. Pasareanu profiles; both map to the same OpenAlex Author ID.",
    },
    {
        "source_canonical_researchr_id": "gorelhedin1",
        "canonical_researchr_id": "gorelhedin",
        "canonical_name": "Gorel Hedin",
        "reason": "Same researcher listed as Gorel Hedin and G\u00f6rel Hedin; both map to the same OpenAlex Author ID.",
    },
    {
        "source_canonical_researchr_id": "michellestrout",
        "canonical_researchr_id": "michellemillsstrout",
        "canonical_name": "Michelle Mills Strout",
        "reason": "Same researcher listed as Michelle Strout and Michelle Mills Strout; both map to the same OpenAlex Author ID.",
    },
])

canonical_id_map = dict(zip(
    researcher_identity_merges["source_canonical_researchr_id"],
    researcher_identity_merges["canonical_researchr_id"],
))
canonical_name_map = dict(zip(
    researcher_identity_merges["canonical_researchr_id"],
    researcher_identity_merges["canonical_name"],
))

pc_members["source_canonical_researchr_id"] = pc_members["canonical_researchr_id"]
pc_members["canonical_researchr_id"] = (
    pc_members["canonical_researchr_id"].replace(canonical_id_map)
)
pc_members["name"] = (
    pc_members["canonical_researchr_id"].map(canonical_name_map).fillna(pc_members["name"])
)

display(researcher_identity_merges)

pc_members["researcher_id"] = pc_members["canonical_researchr_id"]
pc_members["committee_source"] = "visible_pc_page"
pc_members["is_visible_pc"] = True
pc_members["pc_it"] = True
pc_members["service_key"] = (
    pc_members["conference"].astype(str)
    + "_"
    + pc_members["year"].astype(str)
    + "_"
    + pc_members["researcher_id"].astype(str)
)

final_columns = [
    "service_key",
    "conference",
    "year",
    "researcher_id",
    "name",
    "role",
    "affiliation",
    "country",
    "person_url",
    "researchr_id",
    "source_canonical_researchr_id",
    "canonical_researchr_id",
    "committee_source",
    "source",
    "is_visible_pc",
    "pc_it",
    "source_url",
    "final_url",
]

pc_members = (
    pc_members[final_columns]
    .sort_values(["conference", "year", "name"])
    .reset_index(drop=True)
)

display(pc_members.head())


,source_canonical_researchr_id,canonical_researchr_id,canonical_name,reason
0,guoqingharryxu,harryxu,Guoqing Harry Xu,Same researcher listed as Harry Xu on PLDI 201...
1,vikashmansinghka1,vikashmansinghka,Vikash K. Mansinghka,Same researcher listed as Vikash Mansinghka an...
2,corinapasareanu,corinaspasareanu,Corina S. Pasareanu,Same researcher listed as Corina Pasareanu and...
3,corinaspasareanu1,corinaspasareanu,Corina S. Pasareanu,Same researcher listed with accented and unacc...
4,gorelhedin1,gorelhedin,Gorel Hedin,Same researcher listed as Gorel Hedin and Göre...
5,michellestrout,michellemillsstrout,Michelle Mills Strout,Same researcher listed as Michelle Strout and ...


,service_key,conference,year,researcher_id,name,role,affiliation,country,person_url,researchr_id,source_canonical_researchr_id,canonical_researchr_id,committee_source,source,is_visible_pc,pc_it,source_url,final_url
0,ICFP_2017_adamchlipala,ICFP,2017,adamchlipala,Adam Chlipala,PC Member,"Massachusetts Institute of Technology, USA",United States,https://icfp17.sigplan.org/profile/adamchlipala,adamchlipala,adamchlipala,adamchlipala,visible_pc_page,Researchr PC,True,True,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
1,ICFP_2017_alanjeffrey1,ICFP,2017,alanjeffrey1,Alan Jeffrey,PC Member,Mozilla Research,United States,https://icfp17.sigplan.org/profile/alanjeffrey1,alanjeffrey1,alanjeffrey1,alanjeffrey1,visible_pc_page,Researchr PC,True,True,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
2,ICFP_2017_alexandrasilva,ICFP,2017,alexandrasilva,Alexandra Silva,PC Member,University College London,United Kingdom,https://icfp17.sigplan.org/profile/alexandrasilva,alexandrasilva,alexandrasilva,alexandrasilva,visible_pc_page,Researchr PC,True,True,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
3,ICFP_2017_benlippmeier,ICFP,2017,benlippmeier,Ben Lippmeier,PC Member,Digital Asset / UNSW Australia,,https://icfp17.sigplan.org/profile/benlippmeier,benlippmeier,benlippmeier,benlippmeier,visible_pc_page,Researchr PC,True,True,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...
4,ICFP_2017_betaziliani,ICFP,2017,betaziliani,Beta Ziliani,PC Member,"FAMAF, UNC and CONICET",Argentina,https://icfp17.sigplan.org/profile/betaziliani,betaziliani,betaziliani,betaziliani,visible_pc_page,Researchr PC,True,True,https://icfp17.sigplan.org/committee/icfp-2017...,https://icfp17.sigplan.org/committee/icfp-2017...


## 4 - Validate

In [5]:
if pc_members["researcher_id"].isna().any():
    raise ValueError("researcher_id has missing values")

duplicates = pc_members.duplicated(["conference", "year", "researcher_id"], keep=False)
if duplicates.any():
    display(pc_members.loc[duplicates].sort_values(["conference", "year", "researcher_id"]))
    raise ValueError("Duplicate conference-year-researcher rows found")

name_to_id = pc_members.groupby("name")["researcher_id"].nunique()
if (name_to_id > 1).any():
    display(name_to_id[name_to_id > 1])
    raise ValueError("Some display names still map to multiple researcher IDs")

id_to_name = pc_members.groupby("researcher_id")["name"].nunique()
if (id_to_name > 1).any():
    display(id_to_name[id_to_name > 1])
    raise ValueError("Some researcher IDs map to multiple display names")

summary = pd.DataFrame({
    "statistic": [
        "rows",
        "unique researchers",
        "unique raw researchr_id",
        "unique names",
        "conference-year cells",
    ],
    "value": [
        len(pc_members),
        pc_members["researcher_id"].nunique(),
        pc_members["researchr_id"].nunique(),
        pc_members["name"].nunique(),
        pc_members[["conference", "year"]].drop_duplicates().shape[0],
    ],
})

by_conference = (
    pc_members
    .groupby("conference")
    .agg(
        rows=("researcher_id", "size"),
        unique_researchers=("researcher_id", "nunique"),
        years=("year", "nunique"),
    )
    .reset_index()
)

display(summary)
display(by_conference)


,statistic,value
0,rows,2180
1,unique researchers,952
2,unique raw researchr_id,971
3,unique names,952
4,conference-year cells,36


,conference,rows,unique_researchers,years
0,ICFP,318,250,9
1,OOPSLA,524,382,9
2,PLDI,786,480,9
3,POPL,552,384,9


## 5 - Save

In [6]:
pc_members.to_parquet(prepared_dir / "pc_members.parquet", index=False)
researcher_identity_merges.to_csv(
    dependency_tables_dir / "pc_researcher_identity_merges.csv",
    index=False,
)
summary.to_csv(summary_tables_dir / "pc_members_summary.csv", index=False)
by_conference.to_csv(summary_tables_dir / "pc_members_by_conference.csv", index=False)

print(prepared_dir / "pc_members.parquet")
print(dependency_tables_dir / "pc_researcher_identity_merges.csv")
print(pc_members.shape)


/Users/endersari/2026-02-citations-vs-pc-memberships/step_1_data/prepared/pc_members.parquet
/Users/endersari/2026-02-citations-vs-pc-memberships/step_1_artifacts/dependency_tables/pc_researcher_identity_merges.csv
(2180, 18)
